# Amazon rivers-on plume analysis (80 days)

Rivers-on only. Computes surface footprint, centroid displacement, horizontal spreading, and retention within 100 km and 250 km of the Amazon release.

In [ ]:
using Oceananigans
using CairoMakie
using Printf

data_directory = raw"C:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\repo-cleanup\ocean-modeling\amazon_river\new jld2 files\no spinups\80 day salinity dye"
run_prefix = "amazon_rivers_on_validation_FINAL_nz20_gm_off_redi_off_salinity_80day_v2"
dye_file = joinpath(data_directory, "amazon_rivers_on_80d_dye.jld2")
salinity_file = joinpath(data_directory, "amazon_rivers_on_80d_salinity_3d.jld2")

for file in (dye_file, salinity_file)
    @assert isfile(file) "Missing input file: $file. Update data_directory if the 80-day files were moved."
end

dye = FieldTimeSeries(dye_file, "dye"; backend=OnDisk())
salinity = FieldTimeSeries(salinity_file, "S"; backend=OnDisk())
days_saved = Float64.(dye.times) ./ 86400

source_longitude = -49.5
source_latitude = 0.16
println("Loaded $(length(days_saved)) outputs through day $(last(days_saved)).")

In [ ]:
longitude, latitude, depth = nodes(dye.grid, Center(), Center(), Center())
longitude_faces, latitude_faces, depth_faces = nodes(dye.grid, Face(), Face(), Face())
earth_radius = 6.371e6

delta_longitude = diff(deg2rad.(longitude_faces))
delta_sin_latitude = diff(sin.(deg2rad.(latitude_faces)))
delta_depth = diff(depth_faces)

cell_area = earth_radius^2 .* reshape(delta_longitude, length(delta_longitude), 1) .* reshape(delta_sin_latitude, 1, length(delta_sin_latitude))
cell_volume = cell_area .* reshape(delta_depth, 1, 1, length(delta_depth))

initial_salinity = Array(interior(salinity[1]))
wet_cell = isfinite.(initial_salinity) .& (initial_salinity .> 0)
surface_wet = wet_cell[:, :, end]

function haversine_km(longitude_1, latitude_1, longitude_2, latitude_2)
    delta_lambda = deg2rad(longitude_2 - longitude_1)
    delta_phi = deg2rad(latitude_2 - latitude_1)
    phi_1 = deg2rad(latitude_1)
    phi_2 = deg2rad(latitude_2)
    a = sin(delta_phi / 2)^2 + cos(phi_1) * cos(phi_2) * sin(delta_lambda / 2)^2
    return 2 * earth_radius / 1000 * asin(sqrt(clamp(a, 0, 1)))
end

distance_from_source = [haversine_km(source_longitude, source_latitude, lon, lat) for lon in longitude, lat in latitude]
within_100km = wet_cell .& reshape(distance_from_source .<= 100, length(longitude), length(latitude), 1)
within_250km = wet_cell .& reshape(distance_from_source .<= 250, length(longitude), length(latitude), 1)

In [ ]:
number_of_times = length(days_saved)
centroid_longitude = zeros(number_of_times)
centroid_latitude = zeros(number_of_times)
centroid_distance_km = zeros(number_of_times)
east_west_spread_km = zeros(number_of_times)
north_south_spread_km = zeros(number_of_times)
fraction_within_100km = zeros(number_of_times)
fraction_within_250km = zeros(number_of_times)

threshold_fractions = (1e-3, 1e-2, 1e-1)
patch_area_km2 = zeros(number_of_times, length(threshold_fractions))

initial_dye = Float64.(Array(interior(dye[1])))
initial_surface_maximum = maximum(initial_dye[:, :, end][surface_wet])

for n in eachindex(days_saved)
    concentration = Float64.(Array(interior(dye[n])))
    concentration[.!isfinite.(concentration)] .= 0
    concentration[.!wet_cell] .= 0

    weighted_volume = concentration .* cell_volume
    total_mass = sum(weighted_volume)
    fraction_within_100km[n] = sum(weighted_volume[within_100km]) / total_mass
    fraction_within_250km[n] = sum(weighted_volume[within_250km]) / total_mass

    surface_concentration = concentration[:, :, end]
    surface_weight = surface_concentration .* cell_area
    surface_mass = sum(surface_weight)

    centroid_longitude[n] = sum(surface_weight .* reshape(longitude, length(longitude), 1)) / surface_mass
    centroid_latitude[n] = sum(surface_weight .* reshape(latitude, 1, length(latitude))) / surface_mass
    centroid_distance_km[n] = haversine_km(source_longitude, source_latitude, centroid_longitude[n], centroid_latitude[n])

    longitude_scale = 111.32 * cosd(centroid_latitude[n])
    east_west_spread_km[n] = sqrt(sum(surface_weight .* ((reshape(longitude, length(longitude), 1) .- centroid_longitude[n]) .* longitude_scale).^2) / surface_mass)
    north_south_spread_km[n] = sqrt(sum(surface_weight .* ((reshape(latitude, 1, length(latitude)) .- centroid_latitude[n]) .* 111.32).^2) / surface_mass)

    for (m, fraction) in enumerate(threshold_fractions)
        mask = surface_wet .& (surface_concentration .>= fraction * initial_surface_maximum)
        patch_area_km2[n, m] = sum(cell_area[mask]) / 1e6
    end
end

In [ ]:
figure = Figure(size=(1200, 900))

area_axis = Axis(figure[1, 1], xlabel="Simulation day", ylabel="Surface area (km²)", title="Detectable surface footprint")
for (m, fraction) in enumerate(threshold_fractions)
    lines!(area_axis, days_saved, patch_area_km2[:, m], linewidth=3, label="≥ $(fraction) × initial maximum")
end
axislegend(area_axis, position=:rt)

distance_axis = Axis(figure[1, 2], xlabel="Simulation day", ylabel="Distance or spread (km)", title="Plume displacement and spreading")
lines!(distance_axis, days_saved, centroid_distance_km, linewidth=3, label="Centroid displacement")
lines!(distance_axis, days_saved, east_west_spread_km, linewidth=3, label="East–west spread")
lines!(distance_axis, days_saved, north_south_spread_km, linewidth=3, label="North–south spread")
axislegend(distance_axis, position=:lt)

retention_axis = Axis(figure[2, 1:2], xlabel="Simulation day", ylabel="Fraction of dye currently in region", title="Local retention around Amazon release")
lines!(retention_axis, days_saved, fraction_within_100km, linewidth=3, label="Within 100 km")
lines!(retention_axis, days_saved, fraction_within_250km, linewidth=3, label="Within 250 km")
axislegend(retention_axis, position=:rt)

figure

In [ ]:
requested_days = [0, 10, 30, 60, 80]
selected_indices = [argmin(abs.(days_saved .- day)) for day in requested_days]
map_figure = Figure(size=(1500, 330))

for (panel, index) in enumerate(selected_indices)
    concentration = Float64.(Array(interior(dye[index])))[:, :, end]
    concentration[.!surface_wet] .= NaN
    plotted_concentration = log10.(max.(concentration, 1e-12))
    axis = Axis(map_figure[1, panel], xlabel="Longitude", ylabel=panel == 1 ? "Latitude" : "", title="Day $(round(days_saved[index], digits=1))")
    heatmap!(axis, longitude, latitude, plotted_concentration; colorrange=(-6, 0), colormap=:viridis)
    scatter!(axis, [source_longitude], [source_latitude], color=:red, marker=:star5, markersize=13)
end
Colorbar(map_figure[1, 6], limits=(-6, 0), colormap=:viridis, label="log₁₀ surface dye")
map_figure